# Localización de la *Magnetic Pileup Boundary* de Marte mediante aprendizaje profundo

**Trabajo final — Machine Learning en Geofluidos**  
**Modelo V14-5D | Misión MAVEN | Instrumentos MAG y SWEA**


## 1. Introducción y objetivos

Marte no posee un campo magnético dipolar global capaz de desviar el viento solar de la misma manera que ocurre en la Tierra. La interacción del viento solar con la atmósfera superior y la ionosfera marcianas da lugar a una **magnetosfera inducida**, delimitada por distintas regiones de plasma. Una de sus fronteras es la *Magnetic Pileup Boundary* (**MPB**), caracterizada por transiciones en las propiedades magnéticas y de las poblaciones de partículas.

En este trabajo estudio la localización temporal de cruces de la MPB a partir de observaciones de la misión **Mars Atmosphere and Volatile EvolutioN (MAVEN)**. Para ello utilizo dos instrumentos complementarios:

- **Magnetometer (MAG):** mide el campo magnético vectorial. Utilizo sus componentes (B_x), (B_y), (B_z) y el módulo (|\mathbf{B}|).
- **Solar Wind Electron Analyzer (SWEA):** mide distribuciones de electrones en función de su energía y dirección de llegada. Para el modelo utilizo una representación espectral promediada angularmente, no la distribución tridimensional completa.

**Mi objetivo principal** es estimar el instante de cruce de la MPB, comparando las predicciones con tiempos identificados previamente en un catálogo. **Mi segundo objetivo** es estudiar las representaciones internas que aprende una red neuronal y evaluar si contienen información relevante sobre la estructura física de los cruces.

Para abordar ambos objetivos, utilizo la arquitectura **V14-5D**, que combina un modelo de referencia de localización inicial (**V7b**) con un refinador temporal basado en dos autoencoders convolucionales. A continuación describo el modelo de referencia, la modificación introducida en V14 y los datos que utilizo para entrenarlo y evaluarlo.

**Alcance del problema.** En este estudio busco localizar una MPB dentro de una vecindad temporal donde ya se conoce la existencia de un cruce catalogado. No evalúo todavía una búsqueda autónoma de todas las fronteras presentes en órbitas completas.

*Fuentes instrumentales: [Misión MAVEN e instrumentos (LASP)](https://lasp.colorado.edu/maven/science/instrument-package/) y [SWEA (NASA)](https://science.nasa.gov/resource/solar-wind-electron-analyzer-for-maven-spacecraft/).*


## 2. Modelo de referencia: V7b

Antes de construir V14, utilicé un modelo denominado **V7b**, diseñado para identificar la MPB como una **transición entre regiones plasmáticas**. En lugar de entrenarlo únicamente para aproximar una función gaussiana alrededor de la anotación temporal, V7b aprende a distinguir estados físicos a ambos lados de un instante candidato y a **ordenar candidatos** según su compatibilidad con un cruce.

### 2.1. Entradas y arquitectura

Para cada instante candidato, V7b procesa una ventana temporal de **600 s** de ambos instrumentos, respetando sus tiempos de adquisición originales.

| Instrumento | Variables de entrada | Representación utilizada |
| --- | --- | --- |
| MAG | (B_x, B_y, B_z, |\mathbf{B}|) | Cuatro canales a cadencia nativa, cercana a 1 s |
| SWEA | Intensidad espectral electrónica | Doce energías logarítmicamente espaciadas entre 30 y 300 eV; cadencia nativa variable |

V7b contiene **dos ramas convolucionales independientes**, una para MAG y otra para SWEA. Cada rama utiliza bloques residuales y produce **16 canales latentes temporales**; este número no corresponde a un vector de 16 coordenadas por cruce, sino a mapas de características que todavía conservan información temporal. Dos decoders reconstruyen las señales de entrada como tarea auxiliar.

Para estudiar el cambio de régimen, V7b agrupa las características latentes medidas **antes de −60 s** y **después de +60 s** respecto del instante candidato. Con esas representaciones construye dos salidas supervisadas:

1. **Clasificación de regiones:** estima, para cada lado, la compatibilidad con magnetovaina (*magnetosheath*) o magnetosfera inducida.
2. **Clasificación de transición:** produce un *logit* que puntúa si el instante candidato es compatible con una MPB, utilizando las representaciones izquierda y derecha y sus diferencias.

Las etiquetas de región son **débiles**: las infiero a partir del orden temporal entre la MPB y el *bow shock* (BS) asociados en el catálogo. No las interpreto como identificaciones físicas independientes y perfectas de cada intervalo.

### 2.2. Ranking entre candidatos

La revisión **V7b** agrega una tarea que compara varios candidatos pertenecientes al **mismo cruce catalogado**. Durante su entrenamiento considero candidatos positivos próximos a la MPB ((t_{\mathrm{MPB}}-30\,\mathrm{s}), (t_{\mathrm{MPB}}) y (t_{\mathrm{MPB}}+30\,\mathrm{s})) y controles alejados que superan filtros físicos de seguridad.

Además de clasificar cada candidato, penalizo que un control negativo reciba mayor puntuación que uno positivo. La pérdida de ranking tiene la forma

$$
\mathcal{L}_{\mathrm{rank}}
=
\left\langle
\operatorname{softplus}\!\left(m+s_{\mathrm{neg}}-s_{\mathrm{pos}}\right)
\right\rangle,
$$

donde (s_{\mathrm{pos}}) y (s_{\mathrm{neg}}) son los logits de transición y (m=0.5) es el margen utilizado por defecto. También se incorporan **negativos difíciles**, seleccionados entre controles seguros que habían recibido puntuaciones altas durante el entrenamiento previo.

La función de pérdida de referencia combina clasificación de transición, clasificación de regiones, reconstrucción y ranking:

$$
\mathcal{L}_{\mathrm{V7b}}
=
\mathcal{L}_{\mathrm{BCE,trans}}
+0.5\,\mathcal{L}_{\mathrm{BCE,reg}}
+0.15\,(\mathcal{L}_{\mathrm{rec,MAG}}+\mathcal{L}_{\mathrm{rec,SWEA}})
+\mathcal{L}_{\mathrm{rank}}.
$$

### 2.3. Localización inicial

Para localizar un evento, evalúo los puntajes de transición de V7b sobre ventanas con centros distribuidos en **±30 min** alrededor del cruce catalogado, con una separación de **32 s** entre centros candidatos. El instante que obtiene la mayor puntuación constituye mi **estimación inicial**:

$$
t_{\mathrm{coarse}}=\underset{t\;\mathrm{candidato}}{\arg\max}\;s_{\mathrm{V7b}}(t).
$$

En V14 **conservo el modelo V7b previamente entrenado y mantengo congelados sus parámetros**: lo utilizo para proponer el tiempo inicial, mientras que entreno por separado el refinador encargado de corregirlo.


## 3. Modelo propuesto: V14-5D

El objetivo de V14 es **refinar la estimación temporal de V7b** y, simultáneamente, obtener un espacio latente compacto que pueda analizar físicamente.

Diseñé dos **autoencoders convolucionales independientes**, uno para MAG y otro para SWEA. Cada encoder tiene **ocho capas Conv1D** y comprime una ventana de diez minutos en un vector de **cinco coordenadas escalares**:

$$
\mathbf{z}_{\mathrm{MAG}}\in\mathbb{R}^{5},
\qquad
\mathbf{z}_{\mathrm{SWEA}}\in\mathbb{R}^{5}.
$$

Cada decoder intenta reconstruir su señal normalizada a partir de esas cinco coordenadas. De esta forma, incorporo una tarea de reconstrucción para favorecer que el espacio latente conserve información de las observaciones y no dependa exclusivamente del objetivo temporal.

Para calcular la corrección de tiempo, concateno los **cinco valores latentes MAG**, los **cinco SWEA**, el **logit de transición** de V7b y sus **dos logits de región**: en total, **13 características**. Una cabeza de refinamiento produce una distribución temporal sobre muestras MAG próximas al candidato inicial (hasta ±250 s). A partir de ella obtengo una corrección usando su media o su máximo.

Finalmente, calculo

$$
t_{\mathrm{V14}}=t_{\mathrm{coarse}}+\alpha\,\Delta t,
$$

donde la corrección aplicada se limita a **±240 s** y el factor (\alpha) se calibra utilizando exclusivamente un subconjunto interno de entrenamiento (*monitor*), nunca el fold externo de evaluación.

Entreno el refinador con una combinación de clasificación auxiliar de presencia, localización temporal, regresión del desplazamiento y reconstrucción:

$$
\mathcal{L}_{\mathrm{V14}}
=
\mathcal{L}_{\mathrm{BCE}}
+3\,\mathcal{L}_{\mathrm{KL,temporal}}
+1.5\,\mathcal{L}_{\mathrm{SmoothL1,offset}}
+0.2\bigl(
\mathcal{L}_{\mathrm{rec,MAG}}+
\mathcal{L}_{\mathrm{rec,SWEA}}
\bigr).
$$

Para construir la referencia temporal de la pérdida KL utilizo una gaussiana con (\sigma=24\,\mathrm{s}). Este parámetro define la anchura de una **etiqueta de entrenamiento** y no representa el espesor físico de la MPB.

La principal diferencia respecto de V7b es que **los encoders nuevos de V14 sí entregan un vector de cinco números por ventana**, que podré estudiar directamente o proyectar a dos o tres dimensiones mediante PCA. Sin embargo, la posible mejora de localización y la interpretabilidad física de esos vectores deben evaluarse con datos: no las doy por demostradas de antemano.


## 4. Datos de entrenamiento y validación

### 4.1. Catálogo de cruces

Parto de un catálogo de **569 cruces MPB** anotados entre **2014 y 2020**. Para cada evento dispongo de un instante de referencia (t_{\mathrm{MPB}}), información de procedencia y una asociación temporal con un cruce de *bow shock*. El catálogo reúne cuatro grupos de anotaciones; **estos grupos indican procedencia y no corresponden necesariamente a cuatro clases físicas**.

La disponibilidad temporal previamente comprobada cubre **569 eventos en MAG** y **559 en SWEA**. Como V14 utiliza los dos instrumentos de forma conjunta, selecciono los **559 cruces con cobertura temporal SWEA identificada en el manifiesto**. Esta cifra representa la muestra potencial inicial: una ventana concreta todavía puede descartarse si presenta datos incompletos, discontinuidades o asociaciones físicas ambiguas. Por lo tanto, distingo entre *eventos catalogados*, *eventos con cobertura declarada* y *eventos finalmente evaluables*.

### 4.2. Variables y preprocesamiento

Construyo ventanas de **600 s** alrededor de los instantes candidatos. Los datos se preparan de forma independiente para MAG y SWEA:

| Aspecto | MAG | SWEA |
| --- | --- | --- |
| Variables | (B_x,B_y,B_z,|\mathbf B|) | Espectro electrónico promediado angularmente |
| Canales del modelo | 4 | 12 energías entre 30 y 300 eV |
| Muestreo temporal | Cercano a 1 muestra/s | Cadencia propia, variable según el intervalo |
| Tratamiento temporal | Conservo tiempos y muestras válidas | **No interpolo en el tiempo** |
| Preparación adicional | Normalización por canal | Transformación logarítmica del flujo, interpolación **solo en energía** y normalización por canal |

Para SWEA construyo una grilla de doce energías espaciadas logarítmicamente entre 30 y 300 eV. La interpolación se realiza **en el eje energético**, dentro del rango observado, y no crea nuevos tiempos de medición. Como las ventanas pueden contener cantidades diferentes de muestras, uso **relleno (*padding*) y máscaras**, de modo que los valores artificiales no contribuyan a la pérdida de reconstrucción.

Para evitar introducir información del conjunto de evaluación en el entrenamiento, utilizo los **normalizadores del checkpoint V7b correspondiente a cada fold**, calculados originalmente solo a partir de sus datos de entrenamiento internos. Los datos físicos se leen de bloques almacenados en caché local; no necesito volver a descargar todas las series para cada ventana.

### 4.3. Construcción de ejemplos para V14

Para cada cruce con datos válidos, preparo ejemplos de dos tipos:

- **Ventanas positivas:** desplazo el centro respecto de la MPB catalogada en (-240,-180,-120,-60,0,60,120,180,240\,\mathrm{s}). La frontera puede aparecer en distintas posiciones dentro de la ventana, de modo que el modelo no aprenda que siempre coincide con su centro.
- **Controles alejados:** considero centros a (pm600,pm900,pm1200,pm1500\,\mathrm{s}), siempre que cumplan las reglas de exclusión respecto del *bow shock* y de otras fronteras catalogadas, y que las mediciones sean válidas.

Las ventanas positivas reciben una etiqueta de presencia y un **desplazamiento temporal verdadero** respecto del centro. A los controles negativos **no les asigno un tiempo de MPB ficticio**: sus pérdidas de localización y desplazamiento no contribuyen como si existiera una frontera en esas ventanas.

Por época, el entrenamiento de V14 selecciona **hasta seis positivos y tres controles por evento**. Rechazo ventanas con cobertura insuficiente, muestras inválidas o discontinuidades temporales grandes. Los controles constituyen negativos **según el catálogo y los criterios físicos empleados**, no una demostración de ausencia de toda frontera no anotada.

### 4.4. Separación temporal de entrenamiento y evaluación

Utilizo **validación cruzada externa de cuatro folds**, agrupando los eventos por **mes calendario UTC**. Todas las ventanas y los controles de un mismo evento permanecen en el fold que le corresponde: no divido aleatoriamente ventanas superpuestas entre entrenamiento y validación.

La distribución de los **559 eventos con cobertura SWEA declarada** es:

| Fold externo | Eventos conjuntos disponibles para entrenamiento | Eventos conjuntos disponibles para evaluación |
| --- | ---: | ---: |
| 1 | 413 | 146 |
| 2 | 420 | 139 |
| 3 | 424 | 135 |
| 4 | 420 | 139 |

En cada iteración, reservo el fold indicado para la **evaluación externa**. Dentro de los otros tres folds separo, también por meses, un subconjunto **FIT** para ajustar los parámetros del refinador y otro **MONITOR** para seleccionar la época mediante *early stopping* y calibrar la corrección temporal. En el entrenamiento completo utilizo un máximo de **150 épocas**, una paciencia de **20 épocas**, un *batch size* de **16** y una tasa de aprendizaje inicial de (10^{-4}).

No utilizo las anotaciones del fold externo para ajustar los pesos ni para seleccionar (\alpha). Las comparaciones entre versiones del modelo, desarrolladas de manera exploratoria, requieren de todos modos prudencia al interpretar el desempeño como una estimación independiente de generalización.

### 4.5. Criterios de evaluación

Mido el error de localización de cada cruce mediante

$$
\epsilon_i=\left|t_{\mathrm{pred},i}-t_{\mathrm{MPB},i}\right|.
$$

Informaré la **mediana** de (\epsilon_i), su distribución y las fracciones de eventos con (\epsilon_i\le60\,\mathrm{s}) y (\epsilon_i\le120\,\mathrm{s}), calculadas sobre los eventos efectivamente evaluables. También analizaré por separado los espacios latentes MAG y SWEA: los encoders de distintos folds se entrenan con parámetros diferentes, por lo que **sus ejes latentes no pueden mezclarse directamente** sin un procedimiento de alineación.

El análisis de las ventanas centradas en el tiempo real de la MPB será **retrospectivo**; para caracterizar las predicciones sin usar ese tiempo como entrada, también consideraré ventanas centradas en el instante estimado por V14.
